# 🕸️ SNA Analysis: Instagram Social Network

**Input:** Processed files dari notebook 01_EDA.ipynb

**Pipeline:**
1. Load & Build Network Graph
2. Network Overview Statistics
3. Centrality Metrics Computation
4. Influencer Tier Ranking
5. Community Detection (Louvain)
6. White Space Analysis
7. Interactive Visualization (Pyvis)
8. Export untuk Backend API

In [1]:
# ── Install dependencies (jalankan sekali) ─────────────────────────────────
!pip install networkx python-louvain pyvis pandas numpy matplotlib seaborn scipy


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import json
import warnings
from collections import defaultdict, Counter
from pyvis.network import Network
import community as community_louvain  # python-louvain
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

print('✅ Libraries loaded')

✅ Libraries loaded


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Edge architecture:
#   PRIMARY  : comment-on-post  (user A comment di post user B → edge A→B)
#   LAYER 2  : mention          (boost weight jika A juga mention B)
#   SKIP     : co-keyword       (simpan sebagai node attribute, bukan edge)

MENTION_WEIGHT_BOOST    = 1.5    # Multiplier weight jika mention + comment pada edge yang sama
MIN_EDGE_WEIGHT         = 1      # Filter edges di bawah threshold ini
MIN_NODE_POSTS          = 1      # Filter users dengan post < threshold
TOP_N_NODES_VIZ         = 500    # Limit untuk visualisasi (supaya tidak overloaded)
PAGERANK_ALPHA          = 0.85   # Damping factor PageRank (default 0.85)
LOUVAIN_RESOLUTION      = 1.0    # Louvain resolution (>1 = more communities, <1 = fewer)
LOUVAIN_RANDOM_STATE    = 42

print('✅ Config set')
print(f'  Mention weight boost : {MENTION_WEIGHT_BOOST}x')
print(f'  Min edge weight      : {MIN_EDGE_WEIGHT}')
print(f'  Viz node limit       : {TOP_N_NODES_VIZ}')

Config set ✅


In [5]:
# ── Load Raw Dataset ───────────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/dataset_final_clean_revised.csv', low_memory=False)
print(f'Raw dataset shape : {df.shape}')
print(f'Columns           : {list(df.columns)}')

Raw dataset shape : (2215, 20)
Columns           : ['date', 'keyword', 'url', 'content', 'username', 'total_like', 'total_interaction', 'content_processed', 'sentiment_label', 'sentiment_score', 'post_id', 'comment_id', 'is_comment', 'year_month', 'day_of_week', 'hour', 'mentions', 'hashtags', 'n_mentions', 'n_hashtags']


In [6]:
# ── Load user_with_comment_in_post (PRIMARY edge source) ──────────────────────
# Kolom yang dibutuhkan:
#   commenter_username  : user yang komentar (source node)
#   post_owner_username : user pemilik post  (target node)
#   comment_count       : berapa kali A komentar di post milik B (edge weight)
#   post_count, total_interaction, avg_interaction, total_like, total_view : node attrs

comment_edges_raw = pd.read_csv('../data/processed/user_with_comment_in_post.csv')
print(f'comment_in_post shape : {comment_edges_raw.shape}')
print(f'Columns               : {list(comment_edges_raw.columns)}')
print()
print(comment_edges_raw.head(3).to_string())

comment_in_post shape : (1642, 10)
Columns               : ['username', 'activity_count', 'post_id_count', 'post_id_list', 'comment_id_count', 'comment_id_list', 'post_count', 'comment_count', 'total_interaction', 'total_like']

     username  activity_count  post_id_count     post_id_list  comment_id_count        comment_id_list  post_count  comment_count  total_interaction  total_like
0  ronnypunya               1              1  ['DV-MbYoERU5']                 1  ['17878985490522444']           0              1                131        1982
1    bi_lang_               1              1  ['DV-MbYoERU5']                 1  ['17963251245055932']           0              1                 95        1009
2   ricky6510               1              1  ['DVf8X92Dxht']                 1  ['17927179752235370']           0              1                 84         505


---
## 1. Build Network Graph

In [ ]:
# ── Combine & Normalize Edges ─────────────────────────────────────────────────
all_edges = []

if USE_MENTION_EDGES and len(edges_mention) > 0:
    em = edges_mention.copy()
    em['edge_type'] = 'mention'
    em['weight'] = em.get('weight', 1)
    all_edges.append(em[['source', 'target', 'weight', 'edge_type']])

if USE_COKW_EDGES and len(edges_cokw) > 0:
    ek = edges_cokw.copy()
    # Normalize weight to [0,1] range based on avg_interaction
    ek['weight'] = ek.get('co_keyword_count', 1)
    ek['edge_type'] = 'co_keyword'
    all_edges.append(ek[['source', 'target', 'weight', 'edge_type']])

if not all_edges:
    raise ValueError('❌ No edges available! Check EDA output files.')

edges_combined = pd.concat(all_edges, ignore_index=True)

# Aggregate if same source-target appears in multiple edge types
edges_final = edges_combined.groupby(['source', 'target']).agg(
    weight=('weight', 'sum'),
    edge_types=('edge_type', lambda x: '+'.join(sorted(set(x))))
).reset_index()

# Filter by minimum weight
edges_final = edges_final[edges_final['weight'] >= MIN_EDGE_WEIGHT]
print(f'Final edges: {len(edges_final):,}')

In [ ]:
# ── Build DiGraph (Directed) ───────────────────────────────────────────────────
G_directed = nx.DiGraph()

# Add nodes with attributes
user_attr = user_metrics.set_index('username').to_dict('index')

for u, attrs in user_attr.items():
    G_directed.add_node(
        u,
        post_count=int(attrs.get('post_count', 0)),
        total_interaction=float(attrs.get('total_interaction', 0) or 0),
        avg_interaction=float(attrs.get('avg_interaction', 0) or 0),
        total_like=float(attrs.get('total_like', 0) or 0),
        total_view=float(attrs.get('total_view', 0) or 0),
    )

# Add edges
for _, row in edges_final.iterrows():
    G_directed.add_edge(
        row['source'], row['target'],
        weight=float(row['weight']),
        edge_type=row['edge_types']
    )

# Also create undirected version for community detection
G_undirected = G_directed.to_undirected()
# Merge parallel edges by summing weight
for u, v, data in G_undirected.edges(data=True):
    if G_directed.has_edge(v, u):
        data['weight'] = data.get('weight', 1) + G_directed[v][u].get('weight', 1)

print(f'Directed graph   — Nodes: {G_directed.number_of_nodes():,}, Edges: {G_directed.number_of_edges():,}')
print(f'Undirected graph — Nodes: {G_undirected.number_of_nodes():,}, Edges: {G_undirected.number_of_edges():,}')

---
## 2. Network Overview Statistics

In [ ]:
# ── Network Stats ──────────────────────────────────────────────────────────────
N = G_directed.number_of_nodes()
E = G_directed.number_of_edges()

density = nx.density(G_directed)
avg_degree = E / N if N > 0 else 0

# Connected components (undirected)
components = list(nx.connected_components(G_undirected))
largest_cc = max(components, key=len)
lcc_ratio = len(largest_cc) / N

print('=' * 55)
print('  NETWORK OVERVIEW')
print('=' * 55)
print(f'  Nodes (users)        : {N:,}')
print(f'  Edges (interactions) : {E:,}')
print(f'  Density              : {density:.6f}')
print(f'  Avg degree           : {avg_degree:.2f}')
print(f'  Connected components : {len(components):,}')
print(f'  Largest CC (LCC) size: {len(largest_cc):,} ({lcc_ratio*100:.1f}% of nodes)')
print('=' * 55)

# Degree distribution
in_degrees  = dict(G_directed.in_degree())
out_degrees = dict(G_directed.out_degree())
all_degrees = dict(G_undirected.degree())

print(f'  Avg in-degree  : {np.mean(list(in_degrees.values())):.2f}')
print(f'  Avg out-degree : {np.mean(list(out_degrees.values())):.2f}')
print(f'  Max in-degree  : {max(in_degrees.values())} (most-mentioned)')
print(f'  Max out-degree : {max(out_degrees.values())} (most active mentioner)')

In [ ]:
# ── Degree Distribution Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, deg_dict, title, color in [
    (axes[0], in_degrees, 'In-Degree', 'steelblue'),
    (axes[1], out_degrees, 'Out-Degree', 'coral'),
    (axes[2], all_degrees, 'Total Degree (Undirected)', 'mediumseagreen')
]:
    vals = list(deg_dict.values())
    vals_nonzero = [v for v in vals if v > 0]
    if vals_nonzero:
        ax.hist(np.log1p(vals_nonzero), bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{title} Distribution (log1p)')
    ax.set_xlabel('log1p(degree)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/processed/sna_degree_dist.png', bbox_inches='tight')
plt.show()

---
## 3. Centrality Metrics Computation

In [ ]:
# ── Compute All Centrality Measures ───────────────────────────────────────────
print('Computing centrality metrics... (may take a moment for large graphs)')

# For large graphs, use LCC subgraph for expensive metrics
G_lcc = G_undirected.subgraph(largest_cc).copy()
G_lcc_dir = G_directed.subgraph(largest_cc).copy()

# 1. Degree Centrality
print('  [1/6] Degree centrality...')
degree_centrality = nx.degree_centrality(G_undirected)
in_degree_centrality = nx.in_degree_centrality(G_directed)
out_degree_centrality = nx.out_degree_centrality(G_directed)

# 2. Betweenness Centrality (expensive, use sample for large graphs)
print('  [2/6] Betweenness centrality...')
if N > 1000:
    print(f'       Graph large ({N} nodes), using k=500 sample')
    betweenness = nx.betweenness_centrality(G_undirected, k=500, weight='weight', normalized=True)
else:
    betweenness = nx.betweenness_centrality(G_undirected, weight='weight', normalized=True)

# 3. Closeness Centrality
print('  [3/6] Closeness centrality...')
closeness = nx.closeness_centrality(G_lcc)
# Extend to full graph (non-LCC nodes get 0)
closeness_full = {node: closeness.get(node, 0.0) for node in G_undirected.nodes()}

# 4. PageRank (treats it as directed)
print('  [4/6] PageRank...')
pagerank = nx.pagerank(G_directed, alpha=0.85, weight='weight', max_iter=200)

# 5. Eigenvector Centrality
print('  [5/6] Eigenvector centrality...')
try:
    eigenvector = nx.eigenvector_centrality(G_undirected, max_iter=1000, weight='weight')
except nx.PowerIterationFailedConvergence:
    print('       ⚠️  Eigenvector did not converge, using approximate')
    eigenvector = nx.eigenvector_centrality_numpy(G_undirected, weight='weight')
eigenvector = {n: eigenvector.get(n, 0.0) for n in G_undirected.nodes()}

# 6. HITS (Hubs & Authorities) - specific to directed networks
print('  [6/6] HITS (hubs & authorities)...')
try:
    hubs, authorities = nx.hits(G_directed, max_iter=500)
except:
    hubs = {n: 0.0 for n in G_directed.nodes()}
    authorities = {n: 0.0 for n in G_directed.nodes()}

print('✅ All centrality metrics computed!')

In [ ]:
# ── Compile Centrality DataFrame ───────────────────────────────────────────────
all_nodes = list(G_directed.nodes())

centrality_df = pd.DataFrame({
    'username': all_nodes,
    'degree_centrality': [degree_centrality.get(n, 0) for n in all_nodes],
    'in_degree_centrality': [in_degree_centrality.get(n, 0) for n in all_nodes],
    'out_degree_centrality': [out_degree_centrality.get(n, 0) for n in all_nodes],
    'betweenness_centrality': [betweenness.get(n, 0) for n in all_nodes],
    'closeness_centrality': [closeness_full.get(n, 0) for n in all_nodes],
    'pagerank': [pagerank.get(n, 0) for n in all_nodes],
    'eigenvector_centrality': [eigenvector.get(n, 0) for n in all_nodes],
    'hub_score': [hubs.get(n, 0) for n in all_nodes],
    'authority_score': [authorities.get(n, 0) for n in all_nodes],
    'in_degree': [G_directed.in_degree(n) for n in all_nodes],
    'out_degree': [G_directed.out_degree(n) for n in all_nodes],
    'total_degree': [G_undirected.degree(n) for n in all_nodes],
})

# Merge with user metrics
centrality_df = centrality_df.merge(user_metrics, on='username', how='left')

print(f'Centrality DataFrame shape: {centrality_df.shape}')
print(centrality_df.head(3).to_string())

---
## 4. Influencer Tier Ranking

In [ ]:
# ── Composite Influence Score ──────────────────────────────────────────────────
# Methodology: Weighted composite of multiple centrality measures
# Rationale: Single metric biased → multi-metric composite more robust
#
# Weights (can be tuned based on business context):
#   PageRank       : 30% — captures importance from who mentions you (not just how many)
#   Betweenness    : 25% — captures bridge/broker role in information flow
#   In-Degree      : 20% — raw popularity (how many people mention you)
#   Authority Score: 15% — HITS authority (being pointed to by high-hub users)
#   Eigenvector    : 10% — connected to other important nodes

WEIGHTS = {
    'pagerank':               0.30,
    'betweenness_centrality': 0.25,
    'in_degree_centrality':   0.20,
    'authority_score':        0.15,
    'eigenvector_centrality': 0.10,
}

def min_max_normalize(series):
    min_v, max_v = series.min(), series.max()
    if max_v == min_v:
        return pd.Series(0.0, index=series.index)
    return (series - min_v) / (max_v - min_v)

# Normalize each metric
for col in WEIGHTS.keys():
    if col in centrality_df.columns:
        centrality_df[f'{col}_norm'] = min_max_normalize(centrality_df[col])

# Composite score
centrality_df['influence_score'] = sum(
    centrality_df.get(f'{col}_norm', 0) * w
    for col, w in WEIGHTS.items()
    if f'{col}_norm' in centrality_df.columns
)

# Percentile rank
centrality_df['influence_percentile'] = centrality_df['influence_score'].rank(pct=True) * 100

print('Top 20 by Influence Score:')
top_cols = ['username', 'influence_score', 'influence_percentile',
            'pagerank', 'betweenness_centrality', 'in_degree', 'total_interaction']
top_cols = [c for c in top_cols if c in centrality_df.columns]
print(centrality_df.sort_values('influence_score', ascending=False)[top_cols].head(20).to_string(index=False))

In [ ]:
# ── Tier Assignment ────────────────────────────────────────────────────────────
# Tier Definitions (by percentile):
#   Tier 1 — Mega Influencer  : Top 1%
#   Tier 2 — Macro Influencer : Top 5% (excl T1)
#   Tier 3 — Mid Influencer   : Top 20% (excl T2)
#   Tier 4 — Micro Influencer : Top 50% (excl T3)
#   Tier 5 — Regular User     : Bottom 50%

def assign_tier(percentile):
    if percentile >= 99:   return 1
    elif percentile >= 95: return 2
    elif percentile >= 80: return 3
    elif percentile >= 50: return 4
    else:                  return 5

TIER_LABELS = {
    1: 'Mega Influencer',
    2: 'Macro Influencer',
    3: 'Mid Influencer',
    4: 'Micro Influencer',
    5: 'Regular User'
}

centrality_df['tier'] = centrality_df['influence_percentile'].apply(assign_tier)
centrality_df['tier_label'] = centrality_df['tier'].map(TIER_LABELS)

# Tier distribution
tier_counts = centrality_df['tier_label'].value_counts()
print('Influencer Tier Distribution:')
print(tier_counts.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
tier_order = list(TIER_LABELS.values())
tier_colors = ['#FFD700', '#C0C0C0', '#CD7F32', '#87CEEB', '#D3D3D3']
tier_counts.reindex(tier_order).plot(kind='bar', ax=ax, color=tier_colors, edgecolor='white')
ax.set_title('Influencer Tier Distribution')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../data/processed/sna_tier_distribution.png', bbox_inches='tight')
plt.show()

---
## 5. Community Detection (Louvain)

In [ ]:
# ── Louvain Community Detection ────────────────────────────────────────────────
# Why Louvain?
#   - Best modularity optimization algorithm
#   - O(n log n) time complexity → scalable
#   - Handles weighted graphs
#   - Industry standard for social network communities

print('Running Louvain community detection...')
partition = community_louvain.best_partition(G_undirected, weight='weight', random_state=42)

n_communities = len(set(partition.values()))
modularity = community_louvain.modularity(partition, G_undirected, weight='weight')

print(f'Communities detected : {n_communities}')
print(f'Modularity score     : {modularity:.4f}')
print(f'  (Good modularity: > 0.3, Excellent: > 0.5)')

# Add community to centrality df
centrality_df['community_id'] = centrality_df['username'].map(partition)

# Community sizes
community_sizes = pd.Series(partition).value_counts().sort_values(ascending=False)
print(f'\nCommunity sizes:')
print(community_sizes.head(20).to_string())

In [ ]:
# ── Community Profiling ────────────────────────────────────────────────────────
community_profiles = []

for comm_id, size in community_sizes.items():
    comm_users = centrality_df[centrality_df['community_id'] == comm_id]

    # Top influencer in community
    top_user = comm_users.sort_values('influence_score', ascending=False).iloc[0]

    # Dominant sentiment (from original data)
    comm_posts = df[df['username'].isin(comm_users['username'])]
    if 'sentiment_label' in comm_posts.columns and len(comm_posts) > 0:
        dominant_sentiment = comm_posts['sentiment_label'].mode().iloc[0] if len(comm_posts) > 0 else 'unknown'
    else:
        dominant_sentiment = 'unknown'

    # Dominant keyword
    if 'keyword' in comm_posts.columns and len(comm_posts) > 0:
        dominant_keyword = comm_posts['keyword'].mode().iloc[0] if len(comm_posts) > 0 else 'unknown'
    else:
        dominant_keyword = 'unknown'

    community_profiles.append({
        'community_id': int(comm_id),
        'size': size,
        'top_influencer': top_user['username'],
        'top_influence_score': round(float(top_user['influence_score']), 6),
        'avg_influence_score': round(float(comm_users['influence_score'].mean()), 6),
        'avg_total_interaction': round(float(comm_users['total_interaction'].fillna(0).mean()), 2),
        'dominant_sentiment': dominant_sentiment,
        'dominant_keyword': dominant_keyword,
        'total_posts': len(comm_posts),
    })

community_profiles_df = pd.DataFrame(community_profiles)
print('Community Profiles (Top 10 largest):')
print(community_profiles_df.head(10).to_string(index=False))

---
## 6. White Space Analysis

In [ ]:
# ── White Space Analysis ───────────────────────────────────────────────────────
# White Space = Areas/topics with potential that are currently under-served:
#   1. Keywords with high interaction but few active influencers
#   2. Keywords/topics where sentiment is mostly negative (unmet needs)
#   3. Communities that are highly active but weakly connected to main network
#   4. Time slots with low content but potentially high audience

whitespace_results = {}

if 'keyword' in df.columns:
    # Keyword-level analysis
    kw_analysis = df.groupby('keyword').agg(
        total_posts=('username', 'count'),
        unique_users=('username', 'nunique'),
        avg_interaction=('total_interaction', 'mean'),
        total_interaction=('total_interaction', 'sum'),
        pct_negative=('sentiment_label', lambda x: (x.str.lower()=='negative').mean() * 100),
        pct_positive=('sentiment_label', lambda x: (x.str.lower()=='positive').mean() * 100),
    ).reset_index()

    # Interaction per user (reach efficiency)
    kw_analysis['interaction_per_user'] = kw_analysis['avg_interaction'] / kw_analysis['unique_users'].clip(lower=1)

    print('Keyword-level White Space Analysis:')
    print(kw_analysis.sort_values('interaction_per_user', ascending=False).to_string(index=False))

    whitespace_results['keyword_analysis'] = kw_analysis.to_dict('records')

# Community isolation analysis
isolated_communities = []
for comm_id in community_sizes.index:
    comm_nodes = [n for n, c in partition.items() if c == comm_id]
    if len(comm_nodes) < 2:
        continue
    subgraph = G_undirected.subgraph(comm_nodes)
    # Edges going OUT of community
    internal_edges = subgraph.number_of_edges()
    total_edges = sum(G_undirected.degree(n) for n in comm_nodes) / 2
    isolation_ratio = internal_edges / max(total_edges, 1)
    isolated_communities.append({
        'community_id': int(comm_id),
        'size': community_sizes[comm_id],
        'isolation_ratio': round(isolation_ratio, 4),
        'is_isolated': isolation_ratio > 0.9
    })

isolated_df = pd.DataFrame(isolated_communities)
print(f'\nHighly isolated communities (ratio > 0.9): {(isolated_df["is_isolated"]).sum()}')
print(isolated_df[isolated_df['is_isolated']].to_string(index=False))

whitespace_results['isolated_communities'] = isolated_df.to_dict('records')

---
## 7. Interactive Visualization (Pyvis)

In [ ]:
# ── Build Interactive Network with Pyvis ───────────────────────────────────────
# Limit to TOP_N_NODES most central nodes for performance

top_nodes = centrality_df.sort_values('influence_score', ascending=False).head(TOP_N_NODES)['username'].tolist()
G_viz = G_undirected.subgraph(top_nodes).copy()

# Color map for communities
community_colors = [
    '#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6',
    '#1ABC9C','#E67E22','#34495E','#F1C40F','#16A085',
    '#8E44AD','#27AE60','#D35400','#2980B9','#C0392B',
]

tier_colors_map = {
    1: '#FFD700', 2: '#C0C0C0', 3: '#CD7F32', 4: '#87CEEB', 5: '#D3D3D3'
}

net = Network(
    height='750px', width='100%',
    bgcolor='#1a1a2e', font_color='white',
    directed=False
)
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=100)

# Add nodes
for node in G_viz.nodes():
    row = centrality_df[centrality_df['username'] == node]
    if row.empty:
        continue
    row = row.iloc[0]

    comm_id = int(row.get('community_id', 0) or 0)
    color = community_colors[comm_id % len(community_colors)]

    tier = int(row.get('tier', 5) or 5)
    # Node size based on influence score
    size = 10 + float(row.get('influence_score', 0) or 0) * 60

    title = f"""
    <b>@{node}</b><br>
    Tier: {row.get('tier_label', 'Unknown')}<br>
    Influence Score: {row.get('influence_score', 0):.4f}<br>
    PageRank: {row.get('pagerank', 0):.6f}<br>
    Betweenness: {row.get('betweenness_centrality', 0):.6f}<br>
    Posts: {int(row.get('post_count', 0) or 0)}<br>
    Total Interaction: {int(row.get('total_interaction', 0) or 0):,}<br>
    Community: {comm_id}
    """

    net.add_node(
        node,
        label=node if tier <= 3 else '',  # Show labels only for top tiers
        title=title,
        size=max(size, 5),
        color=color,
        borderWidth=3 if tier == 1 else 1,
        borderWidthSelected=5,
    )

# Add edges
for u, v, data in G_viz.edges(data=True):
    weight = float(data.get('weight', 1))
    net.add_edge(u, v, value=min(weight, 10), color='rgba(255,255,255,0.1)')

# Save
out_path = '../data/processed/network_visualization.html'
net.save_graph(out_path)
print(f'✅ Interactive network saved: {out_path}')
print(f'   Nodes shown: {G_viz.number_of_nodes()}, Edges: {G_viz.number_of_edges()}')

---
## 8. Export untuk Backend API

In [ ]:
# ── Export JSON untuk Backend ──────────────────────────────────────────────────
# Format: node-link JSON (compatible dengan D3.js / react-force-graph)

# Build node list
nodes_export = []
for node in G_directed.nodes():
    row = centrality_df[centrality_df['username'] == node]
    if row.empty:
        continue
    row = row.iloc[0]

    nodes_export.append({
        'id': node,
        'username': node,
        'influence_score': round(float(row.get('influence_score', 0) or 0), 6),
        'tier': int(row.get('tier', 5) or 5),
        'tier_label': str(row.get('tier_label', 'Regular User')),
        'community_id': int(row.get('community_id', 0) or 0),
        'pagerank': round(float(row.get('pagerank', 0) or 0), 8),
        'betweenness': round(float(row.get('betweenness_centrality', 0) or 0), 8),
        'degree': int(row.get('total_degree', 0) or 0),
        'in_degree': int(row.get('in_degree', 0) or 0),
        'out_degree': int(row.get('out_degree', 0) or 0),
        'post_count': int(row.get('post_count', 0) or 0),
        'total_interaction': int(row.get('total_interaction', 0) or 0),
        'avg_interaction': round(float(row.get('avg_interaction', 0) or 0), 2),
    })

# Build edge list
edges_export = []
for u, v, data in G_directed.edges(data=True):
    edges_export.append({
        'source': u,
        'target': v,
        'weight': round(float(data.get('weight', 1)), 4),
        'edge_type': data.get('edge_type', 'unknown')
    })

# Full export
graph_data = {
    'metadata': {
        'total_nodes': G_directed.number_of_nodes(),
        'total_edges': G_directed.number_of_edges(),
        'density': round(nx.density(G_directed), 8),
        'modularity': round(modularity, 6),
        'n_communities': n_communities,
    },
    'nodes': nodes_export,
    'edges': edges_export,
    'communities': community_profiles_df.to_dict('records'),
    'whitespace': whitespace_results,
}

with open('../data/processed/graph_data.json', 'w') as f:
    json.dump(graph_data, f, default=str)
print(f'✅ graph_data.json exported')

# Export influencer ranking CSV
influencer_export = centrality_df.sort_values('influence_score', ascending=False)[[
    'username', 'influence_score', 'influence_percentile', 'tier', 'tier_label',
    'community_id', 'pagerank', 'betweenness_centrality', 'in_degree_centrality',
    'closeness_centrality', 'eigenvector_centrality',
    'in_degree', 'out_degree', 'total_degree',
    'post_count', 'total_interaction', 'avg_interaction'
]]
influencer_export.to_csv('../data/processed/influencer_ranking.csv', index=False)
print(f'✅ influencer_ranking.csv exported ({len(influencer_export)} users)')
community_profiles_df.to_csv('../data/processed/community_profiles.csv', index=False)
print(f'✅ community_profiles.csv exported')

In [ ]:
# ── Final Summary ──────────────────────────────────────────────────────────────
print('''
╔══════════════════════════════════════════════════════════════╗
║                 SNA ANALYSIS COMPLETE                        ║
╚══════════════════════════════════════════════════════════════╝
''')
print(f'  Nodes              : {G_directed.number_of_nodes():,}')
print(f'  Edges              : {G_directed.number_of_edges():,}')
print(f'  Communities        : {n_communities}')
print(f'  Modularity         : {modularity:.4f}')
print(f'  Tier 1 (Mega)      : {(centrality_df["tier"]==1).sum()}')
print(f'  Tier 2 (Macro)     : {(centrality_df["tier"]==2).sum()}')
print(f'  Tier 3 (Mid)       : {(centrality_df["tier"]==3).sum()}')
print('''
  EXPORTS:
  • data/processed/graph_data.json          → Full graph for API
  • data/processed/influencer_ranking.csv   → Influencer tiers
  • data/processed/community_profiles.csv  → Community summary
  • data/processed/network_visualization.html → Interactive viz
''')
print('  NEXT STEP: Load graph_data.json into backend API')